# 04 - Profiling e Qualidade dos Dados - Camada Bronze

## Objetivo

Avaliar a qualidade dos dados ingeridos na camada Bronze antes da aplicação das transformações para a camada Silver.

Serão analisadas as seguintes dimensões:

- Completude
- Consistência
- Unicidade
- Validade
- Possíveis outliers

## Fontes analisadas

- Preço internacional do petróleo Brent - EIA
- Preço de Paridade de Importação (PPI) da gasolina - ANP
- Preço de Paridade de Importação (PPI) do diesel - ANP
- Preços de combustíveis praticados nos postos revendedores - ANP

In [0]:
brent = spark.table("workspace.bronze.brent_raw")
ppi_gasolina = spark.table("workspace.bronze.ppi_gasolina_raw")
ppi_diesel = spark.table("workspace.bronze.ppi_diesel_raw")
precos_anp = spark.table("workspace.bronze.precos_anp_raw")

print("Tabelas Bronze carregadas com sucesso.")

Tabelas Bronze carregadas com sucesso.


In [0]:
print(f"Total de registros: {precos_anp.count():,}")
print(f"Total de colunas: {len(precos_anp.columns)}")

print("\nSchema:")
precos_anp.printSchema()

Total de registros: 1,374,816
Total de colunas: 19

Schema:
root
 |-- Regiao_Sigla: string (nullable = true)
 |-- Estado_Sigla: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Revenda: string (nullable = true)
 |-- CNPJ_da_Revenda: string (nullable = true)
 |-- Nome_da_Rua: string (nullable = true)
 |-- Numero_Rua: string (nullable = true)
 |-- Complemento: string (nullable = true)
 |-- Bairro: string (nullable = true)
 |-- Cep: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Data_da_Coleta: date (nullable = true)
 |-- Valor_de_Venda: string (nullable = true)
 |-- Valor_de_Compra: string (nullable = true)
 |-- Unidade_de_Medida: string (nullable = true)
 |-- Bandeira: string (nullable = true)
 |-- arquivo_origem: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- fonte: string (nullable = true)



In [0]:
display(
    precos_anp
        .groupBy("Produto")
        .count()
        .orderBy("Produto")
)

Produto,count
DIESEL,147954
DIESEL S10,248998
ETANOL,304585
GASOLINA,362015
GASOLINA ADITIVADA,281565
GNV,29699


In [0]:
from pyspark.sql.functions import min, max

display(
    precos_anp.agg(
        min("Data_da_Coleta").alias("data_inicial"),
        max("Data_da_Coleta").alias("data_final")
    )
)

data_inicial,data_final
2025-01-01,2026-08-31


In [0]:
from pyspark.sql.functions import col, sum, when

total_anp = precos_anp.count()

nulos_anp = precos_anp.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in precos_anp.columns
])

display(nulos_anp)

Regiao_Sigla,Estado_Sigla,Municipio,Revenda,CNPJ_da_Revenda,Nome_da_Rua,Numero_Rua,Complemento,Bairro,Cep,Produto,Data_da_Coleta,Valor_de_Venda,Valor_de_Compra,Unidade_de_Medida,Bandeira,arquivo_origem,data_ingestao,fonte
0,0,0,0,0,0,103,1060983,2507,0,0,0,0,1374816,0,0,0,0,0


In [0]:
percentual_nulos_anp = precos_anp.select([
    (
        sum(when(col(c).isNull(), 1).otherwise(0))
        / total_anp * 100
    ).alias(c)
    for c in precos_anp.columns
])

display(percentual_nulos_anp)

Regiao_Sigla,Estado_Sigla,Municipio,Revenda,CNPJ_da_Revenda,Nome_da_Rua,Numero_Rua,Complemento,Bairro,Cep,Produto,Data_da_Coleta,Valor_de_Venda,Valor_de_Compra,Unidade_de_Medida,Bandeira,arquivo_origem,data_ingestao,fonte
0.0,0.0,0.0,0.0,0.0,0.0,0.007491911644903754,77.17272711402835,0.18235167469683217,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0


In [0]:
display(
    precos_anp.select(
        "Valor_de_Venda",
        "Valor_de_Compra"
    ).limit(20)
)

Valor_de_Venda,Valor_de_Compra
"6,15",null
"4,79",null
"5,89",null
"5,99",null
"4,98",null
"6,19",null
"4,99",null
"6,19",null
"6,16",null
"6,29",null


In [0]:
total_registros = precos_anp.count()
total_distintos = precos_anp.drop(
    "arquivo_origem",
    "data_ingestao",
    "fonte"
).distinct().count()

print(f"Total de registros: {total_registros:,}")
print(f"Registros distintos: {total_distintos:,}")
print(f"Possíveis duplicados: {total_registros - total_distintos:,}")

Total de registros: 1,374,816
Registros distintos: 1,374,810
Possíveis duplicados: 6


In [0]:
chave_preco = [
    "CNPJ_da_Revenda",
    "Produto",
    "Data_da_Coleta",
    "Valor_de_Venda"
]

duplicados_chave = (
    precos_anp
        .groupBy(chave_preco)
        .count()
        .filter("count > 1")
)

print(
    f"Combinações repetidas pela chave de negócio: "
    f"{duplicados_chave.count():,}"
)

Combinações repetidas pela chave de negócio: 6


In [0]:
display(
    duplicados_chave
        .orderBy("count", ascending=False)
        .limit(20)
)

CNPJ_da_Revenda,Produto,Data_da_Coleta,Valor_de_Venda,count
07.663.077/0001-90,GASOLINA,2026-02-16,"6,99",2
07.663.077/0001-90,ETANOL,2026-02-18,"5,19",2
07.663.077/0001-90,ETANOL,2026-02-16,"5,19",2
07.663.077/0001-90,GASOLINA,2026-02-18,"6,99",2
07.663.077/0001-90,DIESEL S10,2026-02-18,"5,99",2
07.663.077/0001-90,DIESEL S10,2026-02-16,"5,99",2


In [0]:
from pyspark.sql.functions import col

display(
    precos_anp
        .filter(
            (col("CNPJ_da_Revenda") == "07.663.077/0001-90") &
            (col("Data_da_Coleta").isin("2026-02-16", "2026-02-18"))
        )
        .orderBy("Data_da_Coleta", "Produto")
)

Regiao_Sigla,Estado_Sigla,Municipio,Revenda,CNPJ_da_Revenda,Nome_da_Rua,Numero_Rua,Complemento,Bairro,Cep,Produto,Data_da_Coleta,Valor_de_Venda,Valor_de_Compra,Unidade_de_Medida,Bandeira,arquivo_origem,data_ingestao,fonte


In [0]:
display(
    precos_anp
        .groupBy("Unidade_de_Medida")
        .count()
        .orderBy("count", ascending=False)
)

Unidade_de_Medida,count
R$ / litro,1345117
R$ / m³,29699


In [0]:
display(
    precos_anp
        .groupBy("Estado_Sigla")
        .count()
        .orderBy("Estado_Sigla")
)

Estado_Sigla,count
AC,4661
AL,15072
AM,16731
AP,2668
BA,71127
CE,39824
DF,13154
ES,26825
GO,53615
MA,29062


In [0]:
linhas_duplicadas = (
    precos_anp.alias("p")
    .join(
        duplicados_chave.alias("d"),
        on=[
            "CNPJ_da_Revenda",
            "Produto",
            "Data_da_Coleta",
            "Valor_de_Venda"
        ],
        how="inner"
    )
    .select(
        "CNPJ_da_Revenda",
        "Produto",
        "Data_da_Coleta",
        "Valor_de_Venda",
        "p.Regiao_Sigla",
        "p.Estado_Sigla",
        "p.Municipio",
        "p.Revenda",
        "p.Bandeira",
        "p.arquivo_origem"
    )
    .orderBy(
        "Data_da_Coleta",
        "Produto",
        "arquivo_origem"
    )
)

print(f"Linhas encontradas: {linhas_duplicadas.count()}")

display(linhas_duplicadas)

Linhas encontradas: 12


CNPJ_da_Revenda,Produto,Data_da_Coleta,Valor_de_Venda,Regiao_Sigla,Estado_Sigla,Municipio,Revenda,Bandeira,arquivo_origem
07.663.077/0001-90,DIESEL S10,2026-02-16,"5,99",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-dados-abertos-precos-diesel-gnv.csv
07.663.077/0001-90,DIESEL S10,2026-02-16,"5,99",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-dados-abertos-precos-diesel-gnv.csv
07.663.077/0001-90,ETANOL,2026-02-16,"5,19",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-cados-abertos-preco-gasolina-etanol.csv
07.663.077/0001-90,ETANOL,2026-02-16,"5,19",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-cados-abertos-preco-gasolina-etanol.csv
07.663.077/0001-90,GASOLINA,2026-02-16,"6,99",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-cados-abertos-preco-gasolina-etanol.csv
07.663.077/0001-90,GASOLINA,2026-02-16,"6,99",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-cados-abertos-preco-gasolina-etanol.csv
07.663.077/0001-90,DIESEL S10,2026-02-18,"5,99",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-dados-abertos-precos-diesel-gnv.csv
07.663.077/0001-90,DIESEL S10,2026-02-18,"5,99",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-dados-abertos-precos-diesel-gnv.csv
07.663.077/0001-90,ETANOL,2026-02-18,"5,19",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-cados-abertos-preco-gasolina-etanol.csv
07.663.077/0001-90,ETANOL,2026-02-18,"5,19",S,RS,SANTA MARIA,H. D. PORTELLA & CIA LTDA,IPIRANGA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/02-cados-abertos-preco-gasolina-etanol.csv


In [0]:
from pyspark.sql.functions import regexp_replace, col

precos_anp_profiling = (
    precos_anp
        .withColumn(
            "Valor_de_Venda_num",
            regexp_replace(
                col("Valor_de_Venda"),
                ",",
                "."
            ).cast("double")
        )
)

In [0]:
display(
    precos_anp_profiling.select(
        "Produto",
        "Valor_de_Venda",
        "Valor_de_Venda_num"
    ).limit(20)
)

Produto,Valor_de_Venda,Valor_de_Venda_num
GASOLINA,"6,15",6.15
ETANOL,"4,79",4.79
GASOLINA,"5,89",5.89
GASOLINA ADITIVADA,"5,99",5.99
ETANOL,"4,98",4.98
GASOLINA,"6,19",6.19
ETANOL,"4,99",4.99
GASOLINA,"6,19",6.19
GASOLINA,"6,16",6.16
GASOLINA,"6,29",6.29


In [0]:
valores_invalidos = (
    precos_anp_profiling
        .filter(
            col("Valor_de_Venda").isNotNull() &
            col("Valor_de_Venda_num").isNull()
        )
)

print(
    f"Valores que não puderam ser convertidos: "
    f"{valores_invalidos.count():,}"
)

Valores que não puderam ser convertidos: 0


In [0]:
from pyspark.sql.functions import (
    min,
    max,
    avg,
    percentile_approx
)

estatisticas_precos = (
    precos_anp_profiling
        .groupBy("Produto")
        .agg(
            min("Valor_de_Venda_num").alias("minimo"),
            percentile_approx(
                "Valor_de_Venda_num", 0.5
            ).alias("mediana"),
            avg("Valor_de_Venda_num").alias("media"),
            max("Valor_de_Venda_num").alias("maximo")
        )
        .orderBy("Produto")
)

display(estatisticas_precos)

Produto,minimo,mediana,media,maximo
DIESEL,5.07,6.29,6.395855130649388,9.45
DIESEL S10,4.73,6.39,6.5012064353918495,9.99
ETANOL,2.77,4.49,4.5176044125617665,7.44
GASOLINA,4.99,6.36,6.379880723173885,9.79
GASOLINA ADITIVADA,5.19,6.56,6.57840690426784,9.89
GNV,3.19,4.65,4.686166200882192,6.49


In [0]:
display(
    precos_anp_profiling
        .filter(col("Valor_de_Venda_num") <= 0)
        .select(
            "Produto",
            "Valor_de_Venda_num",
            "Estado_Sigla",
            "Municipio",
            "Revenda",
            "Data_da_Coleta",
            "arquivo_origem"
        )
        .limit(50)
)

Produto,Valor_de_Venda_num,Estado_Sigla,Municipio,Revenda,Data_da_Coleta,arquivo_origem


## Profiling - Brent

In [0]:
print(f"Registros: {brent.count()}")
print(f"Colunas: {len(brent.columns)}")

brent.printSchema()

Registros: 9973
Colunas: 2
root
 |-- Date: timestamp (nullable = true)
 |-- Europe_Brent_Spot_Price_FOB_Dollars_per_Barrel: double (nullable = true)



In [0]:
from pyspark.sql.functions import min, max, avg

display(
    brent.agg(
        min("Date").alias("data_inicial"),
        max("Date").alias("data_final"),
        min("Europe_Brent_Spot_Price_FOB_Dollars_per_Barrel").alias("preco_minimo"),
        avg("Europe_Brent_Spot_Price_FOB_Dollars_per_Barrel").alias("preco_medio"),
        max("Europe_Brent_Spot_Price_FOB_Dollars_per_Barrel").alias("preco_maximo")
    )
)

data_inicial,data_final,preco_minimo,preco_medio,preco_maximo
1987-05-20T00:00:00.000Z,2026-09-09T00:00:00.000Z,9.1,51.4692519803469,143.95


In [0]:
display(
    brent.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in brent.columns
    ])
)

Date,Europe_Brent_Spot_Price_FOB_Dollars_per_Barrel
0,0


In [0]:
duplicados_brent = (
    brent
        .groupBy("Date")
        .count()
        .filter("count > 1")
)

print(f"Datas duplicadas no Brent: {duplicados_brent.count()}")

Datas duplicadas no Brent: 0


## Profiling - PPI Gasolina e Diesel

In [0]:
print("PPI GASOLINA")
print(f"Registros: {ppi_gasolina.count()}")
print(f"Colunas: {len(ppi_gasolina.columns)}")
ppi_gasolina.printSchema()

print("\nPPI DIESEL")
print(f"Registros: {ppi_diesel.count()}")
print(f"Colunas: {len(ppi_diesel.columns)}")
ppi_diesel.printSchema()

PPI GASOLINA
Registros: 409
Colunas: 33
root
 |-- Data: string (nullable = true)
 |-- Manaus: double (nullable = true)
 |-- Itaqui: double (nullable = true)
 |-- Suape: double (nullable = true)
 |-- Aratu: double (nullable = true)
 |-- Santos: double (nullable = true)
 |-- Paranagua: double (nullable = true)
 |-- Tramandai: double (nullable = true)
 |-- Guamare: double (nullable = true)
 |-- Duque_de_Caxias: double (nullable = true)
 |-- Betim: double (nullable = true)
 |-- Cubatao: double (nullable = true)
 |-- Maua: double (nullable = true)
 |-- Paulinia: double (nullable = true)
 |-- Sao_Jose_dos_Campos: double (nullable = true)
 |-- Araucaria: double (nullable = true)
 |-- Canoas: double (nullable = true)
 |-- Manaus_variacao_pct: double (nullable = true)
 |-- Itaqui_variacao_pct: double (nullable = true)
 |-- Suape_variacao_pct: double (nullable = true)
 |-- Aratu_variacao_pct: double (nullable = true)
 |-- Santos_variacao_pct: double (nullable = true)
 |-- Paranagua_variacao_pct:

In [0]:
display(
    ppi_gasolina.select("Data").limit(15)
)

Data
05/11/2018 a 09/11/2018
12/11/2018 a 16/11/2018
19/11/2018 a 23/11/2018
26/11/2018 a 30/11/2018
03/12/2018 a 07/12/2018
10/12/2018 a 14/12/2018
17/12/2018 a 21/12/2018
24/12/2018 a 28/12/2018
31/12/2018 a 04/01/2019
07/01/2019 a 11/01/2019


In [0]:
display(
    ppi_gasolina
        .select("Data")
        .orderBy("Data", ascending=False)
        .limit(15)
)

Data
31/12/2018 a 04/01/2019
31/10/2022 A 04/11/2022
31/08/2026 A 04/09/2026
31/08/2020 A 04/09/2020
31/07/2023 A 04/08/2023
31/05/2021 A 04/06/2021
31/03/2025 A 04/04/2025
31/01/2022 A 04/02/2022
30/12/2024 A 03/01/2025
30/12/2019 A 03/01/2020


In [0]:
display(
    ppi_gasolina.limit(10)
)

Data,Manaus,Itaqui,Suape,Aratu,Santos,Paranagua,Tramandai,Guamare,Duque_de_Caxias,Betim,Cubatao,Maua,Paulinia,Sao_Jose_dos_Campos,Araucaria,Canoas,Manaus_variacao_pct,Itaqui_variacao_pct,Suape_variacao_pct,Aratu_variacao_pct,Santos_variacao_pct,Paranagua_variacao_pct,Tramandai_variacao_pct,Guamare_variacao_pct,Duque_de_Caxias_variacao_pct,Betim_variacao_pct,Cubatao_variacao_pct,Maua_variacao_pct,Paulinia_variacao_pct,Sao_Jose_dos_Campos_variacao_pct,Araucaria_variacao_pct,Canoas_variacao_pct
05/11/2018 a 09/11/2018,null,1.6026040000000001,1.6162379999999998,1.6177739999999998,1.6533779999999998,1.6304540000000003,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
12/11/2018 a 16/11/2018,null,1.5240859999999998,1.537714,1.539262,1.574876,1.551936,null,null,null,null,null,null,null,null,null,null,null,-0.04899401224507138,-0.04858442877843472,-0.04853088255837956,-0.04747976566762102,-0.048157139054521236,null,null,null,null,null,null,null,null,null,null
19/11/2018 a 23/11/2018,null,1.48227,1.4958966666666669,1.4974433333333335,1.5368166666666667,1.5140500000000001,null,null,null,null,null,null,null,null,null,null,null,-0.02743677194069094,-0.02719448046472439,-0.027167997824065315,-0.02416655872166018,-0.02441208915831572,null,null,null,null,null,null,null,null,null,null
26/11/2018 a 30/11/2018,null,1.40253,1.416134,1.4177219999999997,1.458436,1.4356679999999997,null,null,null,null,null,null,null,null,null,null,null,-0.05379586714970952,-0.053320973596661214,-0.05323829727557883,-0.05100196293203485,-0.05176975661305794,null,null,null,null,null,null,null,null,null,null
03/12/2018 a 07/12/2018,null,1.4132,1.4268,1.4284,1.4754,1.4529,null,null,null,null,null,null,null,null,null,null,null,0.007607680406123141,0.007531773123164998,0.0075318010159961535,0.011631638275522604,0.012002775014836597,null,null,null,null,null,null,null,null,null,null
10/12/2018 a 14/12/2018,null,1.4263,1.44,1.4415,1.4892,1.4667,null,null,null,null,null,null,null,null,null,null,null,0.009269742428530847,0.009251471825062918,0.009171100532063825,0.0093533956893046,0.009498244889531104,null,null,null,null,null,null,null,null,null,null
17/12/2018 a 21/12/2018,null,1.354148,1.367754,1.3693440000000001,1.40937,1.386564,null,null,null,null,null,null,null,null,null,null,null,-0.05058683306457268,-0.05017083333333339,-0.05005619146722151,-0.05360596293311848,-0.05463694006954389,null,null,null,null,null,null,null,null,null,null
24/12/2018 a 28/12/2018,null,1.2728526864095433,1.2957004555080813,1.3033278949324882,1.3239864389797171,1.304272036131188,null,null,null,null,null,null,null,null,null,null,null,-0.06003428989331783,-0.05268019285040926,-0.04821002251261319,-0.06058278594001776,-0.05934956040169226,null,null,null,null,null,null,null,null,null,null
31/12/2018 a 04/01/2019,null,1.2621066666666667,1.2758266666666667,1.2771866666666667,1.3119133333333333,1.2892533333333334,null,null,null,null,null,null,null,null,null,null,null,-0.008442469311345735,-0.01533825874408723,-0.020057292080881695,-0.009118753252251977,-0.011515007898508678,null,null,null,null,null,null,null,null,null,null
07/01/2019 a 11/01/2019,null,1.291026,1.3047579999999999,1.306098,1.33958,1.316866,null,null,null,null,null,null,null,null,null,null,null,0.022913541380548796,0.022676539132797435,0.022636732819008465,0.021088791434393572,0.02141756468860523,null,null,null,null,null,null,null,null,null,null


In [0]:
from pyspark.sql.functions import col, regexp_extract, to_date

ppi_gasolina_base = (
    ppi_gasolina
    .withColumn(
        "data_inicio",
        to_date(
            regexp_extract(
                col("Data"),
                r"(\d{2}/\d{2}/\d{4})",
                1
            ),
            "dd/MM/yyyy"
        )
    )
    .withColumn(
        "data_fim",
        to_date(
            regexp_extract(
                col("Data"),
                r"(\d{2}/\d{2}/\d{4}).*(\d{2}/\d{2}/\d{4})",
                2
            ),
            "dd/MM/yyyy"
        )
    )
)

In [0]:
display(
    ppi_gasolina_base.select(
        "Data",
        "data_inicio",
        "data_fim"
    ).limit(20)
)

Data,data_inicio,data_fim
05/11/2018 a 09/11/2018,2018-11-05,2018-11-09
12/11/2018 a 16/11/2018,2018-11-12,2018-11-16
19/11/2018 a 23/11/2018,2018-11-19,2018-11-23
26/11/2018 a 30/11/2018,2018-11-26,2018-11-30
03/12/2018 a 07/12/2018,2018-12-03,2018-12-07
10/12/2018 a 14/12/2018,2018-12-10,2018-12-14
17/12/2018 a 21/12/2018,2018-12-17,2018-12-21
24/12/2018 a 28/12/2018,2018-12-24,2018-12-28
31/12/2018 a 04/01/2019,2018-12-31,2019-01-04
07/01/2019 a 11/01/2019,2019-01-07,2019-01-11


In [0]:
from pyspark.sql.functions import min, max

display(
    ppi_gasolina_base.agg(
        min("data_inicio").alias("data_inicial"),
        max("data_fim").alias("data_final")
    )
)

data_inicial,data_final
2018-11-05,2026-09-04


In [0]:
localidades = [
    "Manaus",
    "Itaqui",
    "Suape",
    "Aratu",
    "Santos",
    "Paranagua",
    "Tramandai",
    "Guamare",
    "Duque_de_Caxias",
    "Betim",
    "Cubatao",
    "Maua",
    "Paulinia",
    "Sao_Jose_dos_Campos",
    "Araucaria",
    "Canoas"
]

print(f"Quantidade de localidades: {len(localidades)}")

Quantidade de localidades: 16


In [0]:
from pyspark.sql.functions import expr, lit, col

stack_expr = ", ".join(
    [f"'{local}', `{local}`" for local in localidades]
)

ppi_gasolina_long = (
    ppi_gasolina_base
    .select(
        "data_inicio",
        "data_fim",
        expr(
            f"stack({len(localidades)}, {stack_expr}) "
            "as (localidade, preco)"
        )
    )
    .withColumn("produto", lit("GASOLINA"))
    .filter(col("preco").isNotNull())
)

In [0]:
display(
    ppi_gasolina_long
    .orderBy("data_inicio", "localidade")
    .limit(30)
)

data_inicio,data_fim,localidade,preco,produto
2018-11-05,2018-11-09,Aratu,1.6177739999999998,GASOLINA
2018-11-05,2018-11-09,Itaqui,1.6026040000000001,GASOLINA
2018-11-05,2018-11-09,Paranagua,1.6304540000000003,GASOLINA
2018-11-05,2018-11-09,Santos,1.6533779999999998,GASOLINA
2018-11-05,2018-11-09,Suape,1.6162379999999998,GASOLINA
2018-11-12,2018-11-16,Aratu,1.539262,GASOLINA
2018-11-12,2018-11-16,Itaqui,1.5240859999999998,GASOLINA
2018-11-12,2018-11-16,Paranagua,1.551936,GASOLINA
2018-11-12,2018-11-16,Santos,1.574876,GASOLINA
2018-11-12,2018-11-16,Suape,1.537714,GASOLINA


In [0]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

janela = Window.orderBy("data_inicio")

validacao_variacao = (
    ppi_gasolina_base
    .select(
        "data_inicio",
        "data_fim",
        col("Itaqui").alias("preco_itaqui"),
        col("Itaqui_variacao_pct").alias("variacao_informada")
    )
    .withColumn(
        "preco_semana_anterior",
        lag("preco_itaqui").over(janela)
    )
    .withColumn(
        "variacao_calculada",
        (col("preco_itaqui") / col("preco_semana_anterior")) - 1
    )
)

display(validacao_variacao.limit(15))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_inicio,data_fim,preco_itaqui,variacao_informada,preco_semana_anterior,variacao_calculada
2018-11-05,2018-11-09,1.6026040000000001,null,null,null
2018-11-12,2018-11-16,1.5240859999999998,-0.04899401224507138,1.6026040000000001,-0.04899401224507138
2018-11-19,2018-11-23,1.48227,-0.02743677194069094,1.5240859999999998,-0.02743677194069094
2018-11-26,2018-11-30,1.40253,-0.05379586714970952,1.48227,-0.05379586714970952
2018-12-03,2018-12-07,1.4132,0.007607680406123141,1.40253,0.007607680406123141
2018-12-10,2018-12-14,1.4263,0.009269742428530847,1.4132,0.009269742428530847
2018-12-17,2018-12-21,1.354148,-0.05058683306457268,1.4263,-0.05058683306457268
2018-12-24,2018-12-28,1.2728526864095433,-0.06003428989331783,1.354148,-0.06003428989331783
2018-12-31,2019-01-04,1.2621066666666667,-0.008442469311345735,1.2728526864095433,-0.008442469311345735
2019-01-07,2019-01-11,1.291026,0.022913541380548796,1.2621066666666667,0.022913541380548796


In [0]:
ppi_diesel_base = (
    ppi_diesel
    .withColumn(
        "data_inicio",
        to_date(
            regexp_extract(
                col("Data"),
                r"(\d{2}/\d{2}/\d{4})",
                1
            ),
            "dd/MM/yyyy"
        )
    )
    .withColumn(
        "data_fim",
        to_date(
            regexp_extract(
                col("Data"),
                r"(\d{2}/\d{2}/\d{4}).*(\d{2}/\d{2}/\d{4})",
                2
            ),
            "dd/MM/yyyy"
        )
    )
)

In [0]:
display(
    ppi_diesel_base.select(
        "Data",
        "data_inicio",
        "data_fim"
    ).limit(10)
)

Data,data_inicio,data_fim
05/11/2018 a 09/11/2018,2018-11-05,2018-11-09
12/11/2018 a 16/11/2018,2018-11-12,2018-11-16
17/11/2018 a 21/11/2018,2018-11-17,2018-11-21
26/11/2018 a 30/11/2018,2018-11-26,2018-11-30
03/12/2018 a 07/12/2018,2018-12-03,2018-12-07
10/12/2018 a 14/12/2018,2018-12-10,2018-12-14
17/12/2018 a 21/12/2018,2018-12-17,2018-12-21
24/12/2018 a 28/12/2018,2018-12-24,2018-12-28
31/12/2018 a 04/01/2019,2018-12-31,2019-01-04
07/01/2019 a 11/01/2019,2019-01-07,2019-01-11


In [0]:
from pyspark.sql.functions import expr, lit, col

pares_gasolina = []

for local in localidades:
    pares_gasolina.append(
        f"'{local}', `{local}`, `{local}_variacao_pct`"
    )

stack_gasolina = ", ".join(pares_gasolina)

ppi_gasolina_long = (
    ppi_gasolina_base
    .select(
        "data_inicio",
        "data_fim",
        expr(
            f"""
            stack(
                {len(localidades)},
                {stack_gasolina}
            ) as (localidade, preco, variacao_semanal)
            """
        )
    )
    .withColumn("produto", lit("GASOLINA"))
    .withColumn(
        "variacao_semanal_pct",
        col("variacao_semanal") * 100
    )
    .filter(col("preco").isNotNull())
)

In [0]:
display(
    ppi_gasolina_long
    .select(
        "data_inicio",
        "data_fim",
        "produto",
        "localidade",
        "preco",
        "variacao_semanal",
        "variacao_semanal_pct"
    )
    .orderBy("data_inicio", "localidade")
    .limit(30)
)

data_inicio,data_fim,produto,localidade,preco,variacao_semanal,variacao_semanal_pct
2018-11-05,2018-11-09,GASOLINA,Aratu,1.6177739999999998,null,null
2018-11-05,2018-11-09,GASOLINA,Itaqui,1.6026040000000001,null,null
2018-11-05,2018-11-09,GASOLINA,Paranagua,1.6304540000000003,null,null
2018-11-05,2018-11-09,GASOLINA,Santos,1.6533779999999998,null,null
2018-11-05,2018-11-09,GASOLINA,Suape,1.6162379999999998,null,null
2018-11-12,2018-11-16,GASOLINA,Aratu,1.539262,-0.04853088255837956,-4.853088255837957
2018-11-12,2018-11-16,GASOLINA,Itaqui,1.5240859999999998,-0.04899401224507138,-4.899401224507138
2018-11-12,2018-11-16,GASOLINA,Paranagua,1.551936,-0.048157139054521236,-4.815713905452124
2018-11-12,2018-11-16,GASOLINA,Santos,1.574876,-0.04747976566762102,-4.747976566762102
2018-11-12,2018-11-16,GASOLINA,Suape,1.537714,-0.04858442877843472,-4.858442877843472


In [0]:
pares_diesel = []

for local in localidades:
    pares_diesel.append(
        f"'{local}', `{local}`, `{local}_variacao_pct`"
    )

stack_diesel = ", ".join(pares_diesel)

ppi_diesel_long = (
    ppi_diesel_base
    .select(
        "data_inicio",
        "data_fim",
        expr(
            f"""
            stack(
                {len(localidades)},
                {stack_diesel}
            ) as (localidade, preco, variacao_semanal)
            """
        )
    )
    .withColumn("produto", lit("DIESEL"))
    .withColumn(
        "variacao_semanal_pct",
        col("variacao_semanal") * 100
    )
    .filter(col("preco").isNotNull())
)

In [0]:
display(
    ppi_diesel_long
    .select(
        "data_inicio",
        "data_fim",
        "produto",
        "localidade",
        "preco",
        "variacao_semanal",
        "variacao_semanal_pct"
    )
    .orderBy("data_inicio", "localidade")
    .limit(30)
)

data_inicio,data_fim,produto,localidade,preco,variacao_semanal,variacao_semanal_pct
2018-11-05,2018-11-09,DIESEL,Aratu,2.264426,null,null
2018-11-05,2018-11-09,DIESEL,Itaqui,2.249686,null,null
2018-11-05,2018-11-09,DIESEL,Paranagua,2.2814099999999997,null,null
2018-11-05,2018-11-09,DIESEL,Santos,2.2999880000000004,null,null
2018-11-05,2018-11-09,DIESEL,Suape,2.2504,null,null
2018-11-12,2018-11-16,DIESEL,Aratu,2.174492,-0.039716025164876156,-3.9716025164876156
2018-11-12,2018-11-16,DIESEL,Itaqui,2.159192,-0.040225169201390854,-4.022516920139085
2018-11-12,2018-11-16,DIESEL,Paranagua,2.191472,-0.03942211176421584,-3.942211176421584
2018-11-12,2018-11-16,DIESEL,Santos,2.210074,-0.03909324744303022,-3.909324744303022
2018-11-12,2018-11-16,DIESEL,Suape,2.160458,-0.039967116956985294,-3.9967116956985294


In [0]:
ppi_silver = (
    ppi_gasolina_long
    .unionByName(ppi_diesel_long)
)

In [0]:
print(f"Registros gasolina: {ppi_gasolina_long.count():,}")
print(f"Registros diesel: {ppi_diesel_long.count():,}")
print(f"Registros totais PPI Silver: {ppi_silver.count():,}")

Registros gasolina: 6,071
Registros diesel: 6,071
Registros totais PPI Silver: 12,142


In [0]:
display(
    ppi_silver
        .groupBy("produto")
        .count()
)

produto,count
GASOLINA,6071
DIESEL,6071


In [0]:
from pyspark.sql.functions import sum, when, col

print("CONTAGEM POR PRODUTO")
ppi_silver.groupBy("produto").count().show()

print("\nNULOS EM CAMPOS ESSENCIAIS")

ppi_silver.select(
    sum(when(col("data_inicio").isNull(), 1).otherwise(0)).alias("data_inicio_nulos"),
    sum(when(col("data_fim").isNull(), 1).otherwise(0)).alias("data_fim_nulos"),
    sum(when(col("produto").isNull(), 1).otherwise(0)).alias("produto_nulos"),
    sum(when(col("localidade").isNull(), 1).otherwise(0)).alias("localidade_nulos"),
    sum(when(col("preco").isNull(), 1).otherwise(0)).alias("preco_nulos")
).show()

CONTAGEM POR PRODUTO
+--------+-----+
| produto|count|
+--------+-----+
|GASOLINA| 6071|
|  DIESEL| 6071|
+--------+-----+


NULOS EM CAMPOS ESSENCIAIS
+-----------------+--------------+-------------+----------------+-----------+
|data_inicio_nulos|data_fim_nulos|produto_nulos|localidade_nulos|preco_nulos|
+-----------------+--------------+-------------+----------------+-----------+
|                0|             0|            0|               0|          0|
+-----------------+--------------+-------------+----------------+-----------+



In [0]:
duplicados_ppi = (
    ppi_silver
        .groupBy(
            "data_inicio",
            "data_fim",
            "produto",
            "localidade"
        )
        .count()
        .filter("count > 1")
)

print(
    f"Combinações duplicadas no PPI Silver: "
    f"{duplicados_ppi.count()}"
)

Combinações duplicadas no PPI Silver: 0


In [0]:
(
    ppi_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.silver.ppi")
)

print("Tabela workspace.silver.ppi criada com sucesso.")

Tabela workspace.silver.ppi criada com sucesso.


In [0]:
ppi_silver_salva = spark.table("workspace.silver.ppi")

print(f"Registros antes da persistência: {ppi_silver.count():,}")
print(f"Registros na tabela Silver: {ppi_silver_salva.count():,}")

Registros antes da persistência: 12,142
Registros na tabela Silver: 12,142


In [0]:
from pyspark.sql.functions import regexp_replace, col

precos_anp_silver = (
    precos_anp
        # remove coluna sem utilidade analítica
        .drop("Valor_de_Compra")

        # converte preço para número
        .withColumn(
            "valor_venda",
            regexp_replace(
                col("Valor_de_Venda"),
                ",",
                "."
            ).cast("double")
        )

        # remove duplicatas exatas da fonte
        .dropDuplicates([
            "Regiao_Sigla",
            "Estado_Sigla",
            "Municipio",
            "Revenda",
            "CNPJ_da_Revenda",
            "Nome_da_Rua",
            "Numero_Rua",
            "Complemento",
            "Bairro",
            "Cep",
            "Produto",
            "Data_da_Coleta",
            "Valor_de_Venda",
            "Unidade_de_Medida",
            "Bandeira"
        ])
)

In [0]:
print(f"Bronze: {precos_anp.count():,}")
print(f"Silver candidata: {precos_anp_silver.count():,}")

Bronze: 1,374,816
Silver candidata: 1,374,810


In [0]:
duplicados_anp = (
    precos_anp
    .groupBy(
        "Regiao_Sigla",
        "Estado_Sigla",
        "Municipio",
        "Revenda",
        "CNPJ_da_Revenda",
        "Nome_da_Rua",
        "Numero_Rua",
        "Complemento",
        "Bairro",
        "Cep",
        "Produto",
        "Data_da_Coleta",
        "Valor_de_Venda",
        "Unidade_de_Medida",
        "Bandeira"
    )
    .count()
    .filter(col("count") > 1)
)

linhas_duplicadas_excedentes = (
    duplicados_anp
    .selectExpr("sum(count - 1) as total")
    .first()["total"]
)

print(f"Grupos com duplicidade na ANP Bronze: {duplicados_anp.count():,}")
print(f"Linhas duplicadas excedentes removidas: {linhas_duplicadas_excedentes:,}")

Grupos com duplicidade na ANP Bronze: 6
Linhas duplicadas excedentes removidas: 6


In [0]:
display(
    precos_anp_silver.select(
        "Produto",
        "Valor_de_Venda",
        "valor_venda"
    ).limit(20)
)

Produto,Valor_de_Venda,valor_venda
GASOLINA,"6,99",6.99
GASOLINA ADITIVADA,"6,79",6.79
GASOLINA ADITIVADA,"7,09",7.09
GASOLINA,"6,99",6.99
GASOLINA,"7,66",7.66
GASOLINA,"7,63",7.63
GASOLINA ADITIVADA,"6,94",6.94
ETANOL,"4,99",4.99
GASOLINA ADITIVADA,"7,38",7.38
GASOLINA,"6,39",6.39


In [0]:
invalidos_silver = (
    precos_anp_silver
        .filter(
            col("Valor_de_Venda").isNotNull() &
            col("valor_venda").isNull()
        )
)

print(
    f"Valores inválidos após conversão: "
    f"{invalidos_silver.count():,}"
)

Valores inválidos após conversão: 0


In [0]:
precos_anp_silver_final = (
    precos_anp_silver
    .select(
        col("Data_da_Coleta").alias("data_coleta"),
        col("Regiao_Sigla").alias("regiao"),
        col("Estado_Sigla").alias("uf"),
        col("Municipio").alias("municipio"),
        col("Produto").alias("produto"),
        col("valor_venda"),
        col("Unidade_de_Medida").alias("unidade_medida"),
        col("Bandeira").alias("bandeira"),
        col("CNPJ_da_Revenda").alias("cnpj_revenda"),
        col("Revenda").alias("revenda"),
        col("arquivo_origem"),
        col("data_ingestao"),
        col("fonte")
    )
)

In [0]:
precos_anp_silver_final.printSchema()

root
 |-- data_coleta: date (nullable = true)
 |-- regiao: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- valor_venda: double (nullable = true)
 |-- unidade_medida: string (nullable = true)
 |-- bandeira: string (nullable = true)
 |-- cnpj_revenda: string (nullable = true)
 |-- revenda: string (nullable = true)
 |-- arquivo_origem: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- fonte: string (nullable = true)



In [0]:
display(
    precos_anp_silver_final.limit(20)
)

data_coleta,regiao,uf,municipio,produto,valor_venda,unidade_medida,bandeira,cnpj_revenda,revenda,arquivo_origem,data_ingestao,fonte
2026-06-01,NE,AL,ARAPIRACA,GASOLINA,6.99,R$ / litro,BRANCA,08.738.994/0001-50,AUTO POSTO M M GARROTE LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,NE,AL,ARAPIRACA,GASOLINA ADITIVADA,6.79,R$ / litro,BRANCA,07.248.398/0001-29,AUTO POSTO MASSARANDUBA LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,NE,AL,ARAPIRACA,GASOLINA ADITIVADA,7.09,R$ / litro,IPIRANGA,08.461.170/0001-85,POSTO ATLANTIC CONFIANCA LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,NE,AL,ARAPIRACA,GASOLINA,6.99,R$ / litro,BRANCA,01.242.690/0001-58,IBN PINTO E SILVA & CIA LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,NE,AL,DELMIRO GOUVEIA,GASOLINA,7.66,R$ / litro,VIBRA,05.518.639/0001-87,AUTO POSTO DA PEDRA LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,NE,AL,DELMIRO GOUVEIA,GASOLINA,7.63,R$ / litro,BRANCA,04.431.113/0001-00,COMERCIO DE COMBUSTIVEIS NENZITA LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,NE,AL,PALMEIRA DOS INDIOS,GASOLINA ADITIVADA,6.94,R$ / litro,BRANCA,05.331.412/0001-28,SEVERINO S. LOPES & CIA. LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,NE,BA,POCOES,ETANOL,4.99,R$ / litro,VIBRA,14.987.317/0001-78,EZENILDA CÉLIA NOVAIS LABANCA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,NE,BA,SALVADOR,GASOLINA ADITIVADA,7.38,R$ / litro,RAIZEN,34.276.766/0001-15,HIPER POSTO CAMINHO DAS ARVORES LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP
2026-06-01,SE,ES,SERRA,GASOLINA,6.39,R$ / litro,VIBRA,07.571.908/0001-02,AUTO POSTO CORAL LTDA,dbfs:/Volumes/workspace/raw/dados_raw/anp_precos/06-dados-abertos-precos-2026-06-gasolina-etanol.csv,2026-09-25T00:55:23.461Z,ANP


In [0]:
(
    precos_anp_silver_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.silver.precos_anp")
)

print("Tabela workspace.silver.precos_anp criada com sucesso.")

Tabela workspace.silver.precos_anp criada com sucesso.


In [0]:
precos_anp_silver_salva = spark.table("workspace.silver.precos_anp")

print(f"Registros antes da persistência: {precos_anp_silver_final.count():,}")
print(f"Registros na tabela Silver: {precos_anp_silver_salva.count():,}")

Registros antes da persistência: 1,374,810
Registros na tabela Silver: 1,374,810


In [0]:
from pyspark.sql.functions import col, to_date

brent_silver = (
    brent
        .select(
            to_date(col("Date")).alias("data"),
            col("Europe_Brent_Spot_Price_FOB_Dollars_per_Barrel")
                .alias("preco_brent_usd")
        )
)

brent_silver.printSchema()

display(
    brent_silver
        .orderBy("data", ascending=False)
        .limit(15)
)

root
 |-- data: date (nullable = true)
 |-- preco_brent_usd: double (nullable = true)



data,preco_brent_usd
2026-09-09,109.51
2026-09-08,106.12
2026-09-07,104.47
2026-09-04,102.24
2026-09-03,100.52
2026-09-02,97.59
2026-09-01,96.02
2026-08-28,89.75
2026-08-27,90.18
2026-08-26,87.77


In [0]:
from pyspark.sql.functions import min, max, avg

print(f"Registros: {brent_silver.count():,}")

display(
    brent_silver.agg(
        min("data").alias("data_inicial"),
        max("data").alias("data_final"),
        min("preco_brent_usd").alias("preco_minimo"),
        avg("preco_brent_usd").alias("preco_medio"),
        max("preco_brent_usd").alias("preco_maximo")
    )
)

Registros: 9,973


data_inicial,data_final,preco_minimo,preco_medio,preco_maximo
1987-05-20,2026-09-09,9.1,51.4692519803469,143.95


In [0]:
(
    brent_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.silver.brent")
)

print("Tabela workspace.silver.brent criada com sucesso.")

Tabela workspace.silver.brent criada com sucesso.


In [0]:
brent_silver_salva = spark.table("workspace.silver.brent")

print(f"Registros antes da persistência: {brent_silver.count():,}")
print(f"Registros na tabela Silver: {brent_silver_salva.count():,}")

Registros antes da persistência: 9,973
Registros na tabela Silver: 9,973
